# Tahap 3 — Arsitektur Model

Verifikasi arsitektur pada [`src/model.py`](../src/model.py), mengikuti Subbab 3.5 naskah proposal.

```
Sinyal (6 kanal) → PatchEmbedding → Encoder (dipertukarkan) → AttentionPooling → Head
```

Notebook ini bukan tempat mendefinisikan model — definisinya ada di modul, karena akan dipakai
berulang oleh tahap-tahap berikutnya. Di sini isinya **bukti bahwa modul itu benar**: anggaran
parameter, penyetaraan antar arsitektur, dan tiga uji korektheses yang tidak akan terdeteksi
lewat metrik akurasi.

In [ ]:
import sys, torch
sys.path.insert(0, "../src")
from model import PDClassifier, BiMamba2Encoder, check_mamba2_config

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", DEV, "|", torch.cuda.get_device_name(0) if DEV == "cuda" else "")

/home/agribychaniago/Python Projects/Skripsi/.venv/lib/python3.11/site-packages/mamba_ssm/ops/selective_scan_interface.py:163: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/agribychaniago/Python Projects/Skripsi/.venv/lib/python3.11/site-packages/mamba_ssm/ops/selective_scan_interface.py:239: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/home/agribychaniago/Python Projects/Skripsi/.venv/lib/python3.11/site-packages/mamba_ssm/ops/triton/layer_norm.py:985: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/agribychaniago/Python Projects/Skripsi/.venv/lib/python3.11/site-packages/mamba_ssm/ops/triton/layer_norm.py:1044: FutureWarning: `torch.cuda.amp.custom_bwd(args

device: cuda | NVIDIA GeForce RTX 3050 6GB Laptop GPU


## 1. Kendala konfigurasi `causal_conv1d`

`causal_conv1d` mensyaratkan `d_in_proj` kelipatan 8, dengan

```
d_in_proj = 2*d_inner + 2*ngroups*d_state + nheads
```

Kendala ini **tidak terdokumentasi** di `mamba_ssm` maupun di makalah — ia hanya muncul sebagai
`RuntimeError` saat forward, bukan saat konstruksi. Konfigurasi `d_model=128, headdim=32`
(dipakai pada smoke test setup awal) kebetulan lolos karena `d_in_proj=648`, sehingga kendala ini
baru ketahuan ketika lebar per-arah dikecilkan menjadi 64.

`check_mamba2_config` memvalidasinya di awal agar gagal cepat dengan pesan yang jelas.

In [ ]:
for hd in [32, 16]:
    try:
        dip = check_mamba2_config(d_model=64, d_state=64, headdim=hd)
        print(f"headdim={hd:2d} -> d_in_proj={dip} (mod 8 = {dip % 8}) DITERIMA")
    except ValueError as e:
        print(f"headdim={hd:2d} -> DITOLAK: {e}")

headdim=32 -> DITOLAK: d_in_proj=388 bukan kelipatan 8 (sisa 4). causal_conv1d akan gagal saat forward. Konfigurasi: d_model=64, d_state=64, headdim=32, expand=2, ngroups=1. Coba headdim lain agar nheads berubah, atau sesuaikan d_state.
headdim=16 -> d_in_proj=392 (mod 8 = 0) DITERIMA


`headdim=16` memberi `nheads=8`, bukan 4. Ini justru menguntungkan: klaim Subbab 2.2.4 soal
pemisahan skala waktu antar head jadi punya lebih banyak head untuk merepresentasikan fenomena
cepat (tremor, ratusan milidetik) dan lambat (penyimpangan bentuk, puluhan detik) secara terpisah.

## 2. Anggaran parameter dan penyetaraan antar arsitektur

Naskah membatasi total parameter di bawah 500 ribu, dan mensyaratkan jumlah parameter kedua
encoder dicatat serta disetarakan sebelum perbandingan dilakukan (Subbab 3.5).

`Mamba2` mengunci dimensi masukan = keluaran = `d_model`, tidak seperti `nn.GRU`. Tiga rakitan
bidirectional yang mungkin, dengan hasil pengukuran:

| Opsi | Susunan | Total | Status |
|---|---|---|---|
| A | 2× Mamba2(128) → concat 256 → proj 128 | 854.929 | jebol |
| B | proj 128→64, 2× Mamba2(64) → concat 128 | 282.577 | dipakai |
| C | 2× Mamba2(128), keluaran dijumlah | 756.241 | jebol |

Opsi B dipilih bukan sekadar karena muat anggaran: proyeksi 128→64 adalah versi eksplisit dari
apa yang GRU lakukan implisit lewat `W_ih`, dan menyamakan lebar state rekuren per arah (64)
antara kedua arsitektur — sehingga perbandingannya setara.

In [ ]:
import pandas as pd

rows = []
for enc in ["mamba2", "gru"]:
    m = PDClassifier(encoder=enc, patch_size=20).to(DEV)
    rows.append({"arsitektur": enc, **m.count_parameters()})

df = pd.DataFrame(rows).set_index("arsitektur")
display(df)

n_mamba, n_gru = df.loc["mamba2", "encoder"], df.loc["gru", "encoder"]
selisih = (n_gru - n_mamba) / n_mamba * 100
print(f"Selisih parameter encoder: {selisih:+.1f}%  -> {'dalam toleransi ±5%' if abs(selisih) <= 5 else 'DI LUAR TOLERANSI'}")
print(f"Total terbesar: {df['total'].max():,} -> {'di bawah 500rb' if df['total'].max() < 500_000 else 'MELEBIHI 500rb'}")

,patch_embed,encoder,pool,head,total
arsitektur,,,,,
mamba2,15488,233808,16640,16641,282577
gru,15488,224256,16640,16641,273025


Selisih parameter encoder: -4.1%  -> dalam toleransi ±5%
Total terbesar: 282,577 -> di bawah 500rb


## 3. Uji korektheses

Tiga uji berikut memeriksa kesalahan yang **tidak terdeteksi lewat metrik akurasi**. Model dengan
bug semacam ini tetap berjalan, tetap menghasilkan angka, dan tetap terlihat masuk akal — tetapi
peta atribusi temporalnya, yang merupakan klaim inti penelitian ini, sudah tercemar.

### 3.1 Kebocoran padding

Uji ini memakai dua tolok ukur yang berbeda kekuatannya, karena keduanya menjawab pertanyaan yang
berbeda.

**Uji A — invariansi terhadap isi padding (uji korektheses sesungguhnya).** Nilai yang ditaruh di
posisi padding tidak boleh memengaruhi keluaran sama sekali. Ini menguji langsung sifat yang
dipedulikan: apakah padding merembes ke data valid. Toleransinya nol mutlak, bukan sekadar kecil.

**Uji B — stabilitas terhadap komposisi batch.** Keluaran satu sampel dibandingkan antara dijalankan
sendirian dan di dalam batch. Uji ini lebih lemah karena mengubah komposisi batch juga mengubah jalur
kernel yang dipilih cuDNN, sehingga urutan akumulasi floating point ikut berubah. Selisih pada orde
1e-5 di float32 adalah derau numerik, bukan kebocoran — dan Uji A yang membuktikannya.

In [ ]:
def uji_A_isi_padding(encoder):
    """Isi padding diganti-ganti; keluaran di posisi valid harus tidak berubah sama sekali."""
    torch.manual_seed(0)
    m = PDClassifier(encoder=encoder, patch_size=20).to(DEV).eval()
    LA, LB = 1880, 1180
    xA = torch.randn(1, LA, 6, device=DEV)
    xB = torch.randn(1, LB, 6, device=DEV)
    lens = torch.tensor([LA, LB], device=DEV)

    logits, alphas = [], []
    with torch.no_grad():
        for isi in [0.0, 1e3, -1e3]:
            pad = torch.full((1, LA - LB, 6), isi, device=DEV)
            lo, al = m(torch.cat([xA, torch.cat([xB, pad], 1)], 0), lens)
            logits.append(lo[1].item())
            alphas.append(al[1, : LB // 20])
    rentang_logit = max(logits) - min(logits)
    rentang_alpha = max((a - alphas[0]).abs().max().item() for a in alphas)
    return {
        "encoder": encoder,
        "rentang_logit": rentang_logit,
        "rentang_alpha": rentang_alpha,
        "lolos": rentang_logit == 0.0 and rentang_alpha == 0.0,
    }


def uji_B_komposisi_batch(encoder):
    """Sampel yang sama, dijalankan solo vs di dalam batch."""
    torch.manual_seed(0)
    m = PDClassifier(encoder=encoder, patch_size=20).to(DEV).eval()
    LA, LB = 1880, 1180
    xA = torch.randn(1, LA, 6, device=DEV)
    xB = torch.randn(1, LB, 6, device=DEV)
    with torch.no_grad():
        solo, a_solo = m(xB, torch.tensor([LB], device=DEV))
        xB_pad = torch.cat([xB, torch.zeros(1, LA - LB, 6, device=DEV)], 1)
        bat, a_bat = m(torch.cat([xA, xB_pad], 0), torch.tensor([LA, LB], device=DEV))
    n = LB // 20
    return {
        "encoder": encoder,
        "selisih_logit": (bat[1] - solo[0]).abs().max().item(),
        "selisih_alpha": (a_bat[1, :n] - a_solo[0, :n]).abs().max().item(),
        "alpha_di_padding": a_bat[1, n:].abs().max().item(),
    }


print("UJI A - invariansi terhadap isi padding (toleransi nol mutlak)")
dfA = pd.DataFrame([uji_A_isi_padding(e) for e in ["mamba2", "gru"]]).set_index("encoder")
display(dfA)
print("LOLOS: padding tidak memengaruhi keluaran sama sekali" if dfA["lolos"].all() else "GAGAL: ADA KEBOCORAN NYATA")

print()
print("UJI B - stabilitas terhadap komposisi batch (toleransi derau float32)")
dfB = pd.DataFrame([uji_B_komposisi_batch(e) for e in ["mamba2", "gru"]]).set_index("encoder")
display(dfB)
print("alpha di posisi padding tepat nol:", (dfB["alpha_di_padding"] == 0.0).all())

UJI A - invariansi terhadap isi padding (toleransi nol mutlak)


,rentang_logit,rentang_alpha,lolos
encoder,,,
mamba2,0.0,0.0,True
gru,0.0,0.0,True


LOLOS: padding tidak memengaruhi keluaran sama sekali

UJI B - stabilitas terhadap komposisi batch (toleransi derau float32)


,selisih_logit,selisih_alpha,alpha_di_padding
encoder,,,
mamba2,7.450581e-09,1.117587e-08,0.0
gru,1.052767e-05,2.659857e-06,0.0


alpha di posisi padding tepat nol: True


Uji A lolos mutlak pada kedua encoder: mengganti isi padding dengan 0, +1000, atau −1000 sama
sekali tidak menggeser keluaran. Inilah bukti korektheses yang sesungguhnya.

Uji B menunjukkan selisih pada orde 1e-8 untuk BiMamba-2 dan 1e-5 untuk BiGRU. Selisih BiGRU yang
lebih besar berasal dari cuDNN yang memilih kernel berbeda ketika ukuran batch berubah, bukan dari
kebocoran informasi — sebagaimana dibuktikan Uji A. Pada skala logit sekitar 1e-2, selisih ini
setara 0,08 persen dan tidak berdampak pada peta atribusi.

### 3.2 Uji regresi arah mundur

Uji yang memotivasi seluruh penanganan padding di `BiMamba2Encoder`. Bila arah mundur diterapkan
dengan `torch.flip` naif atas tensor terpadding, padding berpindah ke **awal** urutan dan diproses
lebih dulu oleh rekurensi SSM, mencemari representasi seluruh posisi valid sesudahnya.

Arah maju tidak memerlukan penanganan khusus — kausalitas melindunginya, karena padding di ujung
tidak dapat memengaruhi posisi sebelumnya.

Uji ini sengaja membuktikan bahwa kedua pendekatan **berbeda**. Bila suatu saat uji ini mulai
menunjukkan hasil yang sama, artinya penanganan padding telah rusak tanpa disadari.

In [ ]:
from mamba_ssm.modules.mamba2 import Mamba2

torch.manual_seed(0)
blok = Mamba2(d_model=64, d_state=64, d_conv=4, expand=2, headdim=16).to(DEV).eval()

LA, LB = 50, 30
hA = torch.randn(1, LA, 64, device=DEV)
hB = torch.randn(1, LB, 64, device=DEV)
batch = torch.cat([hA, torch.cat([hB, torch.zeros(1, LA - LB, 64, device=DEV)], 1)], 0)
lens = torch.tensor([LA, LB], device=DEV)

with torch.no_grad():
    maju_batch = blok(batch)[1, :LB]
    maju_solo = blok(hB)[0]
    naif = torch.flip(blok(torch.flip(batch, [1]))[1], [0])[:LB]
    benar = BiMamba2Encoder._flip_valid(blok(BiMamba2Encoder._flip_valid(batch, lens)), lens)[1, :LB]

skala = benar.abs().max().item()
print(f"ARAH MAJU   batch vs solo      : {(maju_batch - maju_solo).abs().max().item():.3e}  (aman: kausalitas melindungi)")
print(f"ARAH MUNDUR flip-naif vs benar : {(naif - benar).abs().max().item():.3e}  "
      f"({(naif - benar).abs().max().item() / skala * 100:.2f}% dari skala sinyal)")
print()
print("Selisih arah mundur yang besar adalah HASIL YANG DIHARAPKAN dari uji ini:")
print("ia membuktikan flip naif memang mencemari, sehingga penanganan per-panjang-asli diperlukan.")

ARAH MAJU   batch vs solo      : 5.960e-07  (aman: kausalitas melindungi)
ARAH MUNDUR flip-naif vs benar : 1.349e-02  (0.72% dari skala sinyal)

Selisih arah mundur yang besar adalah HASIL YANG DIHARAPKAN dari uji ini:
ia membuktikan flip naif memang mencemari, sehingga penanganan per-panjang-asli diperlukan.


Besarnya kontaminasi bergantung pada banyaknya padding, yang berkorelasi dengan panjang rekaman.
Artinya bug ini bukan derau acak melainkan **bias sistematis yang berkorelasi dengan durasi tugas** —
persis jenis perancu yang akan merusak analisis keselarasan temporal pada Subbab 3.6.6.

### 3.3 Rentang panjang urutan dan kasus tepi

Panjang urutan yang dilihat encoder adalah panjang **setelah** patching, bukan jumlah titik data
mentah. Dihitung dari durasi median hasil verifikasi Tahap 1
([notebook 01](01_verifikasi_struktur_data.ipynb)) pada 142,86 Hz.

In [ ]:
fs = 142.86
durasi = {"STCP": 13.1, "SST": 24.5, "DST": 26.2}
tabel = pd.DataFrame(
    {f"P={P}": {t: int(d * fs) // P for t, d in durasi.items()} for P in [8, 16, 32, 64]}
)
tabel.insert(0, "titik_mentah", {t: int(d * fs) for t, d in durasi.items()})
display(tabel)
print("Satu siklus tremor 7 Hz = 143 ms = 20 sampel pada 142,86 Hz.")
print("Patch di bawah durasi itu tergolong resolusi halus (Subbab 3.5).")

,titik_mentah,P=8,P=16,P=32,P=64
STCP,1871,233,116,58,29
SST,3500,437,218,109,54
DST,3742,467,233,116,58


Satu siklus tremor 7 Hz = 143 ms = 20 sampel pada 142,86 Hz.
Patch di bawah durasi itu tergolong resolusi halus (Subbab 3.5).


In [ ]:
hasil = []
for enc in ["mamba2", "gru"]:
    m = PDClassifier(encoder=enc, patch_size=20).to(DEV).eval()
    for n_patch in [29, 58, 116, 233, 467]:
        T = n_patch * 20
        with torch.no_grad():
            logit, alpha = m(torch.randn(2, T, 6, device=DEV), torch.tensor([T, T - 400], device=DEV))
        hasil.append({
            "arsitektur": enc, "n_patch": n_patch,
            "bentuk_alpha": tuple(alpha.shape),
            "ada_nan": bool(torch.isnan(logit).any() or torch.isnan(alpha).any()),
            "alpha_jumlah_1": bool(torch.allclose(alpha.sum(-1), torch.ones(2, device=DEV), atol=1e-5)),
        })
display(pd.DataFrame(hasil))

,arsitektur,n_patch,bentuk_alpha,ada_nan,alpha_jumlah_1
0,mamba2,29,"(2, 29)",False,True
1,mamba2,58,"(2, 58)",False,True
2,mamba2,116,"(2, 116)",False,True
3,mamba2,233,"(2, 233)",False,True
4,mamba2,467,"(2, 467)",False,True
5,gru,29,"(2, 29)",False,True
6,gru,58,"(2, 58)",False,True
7,gru,116,"(2, 116)",False,True
8,gru,233,"(2, 233)",False,True
9,gru,467,"(2, 467)",False,True


## 4. Forward dan backward penuh

Memastikan gradien mengalir ke seluruh parameter tanpa NaN maupun Inf, pada batch dengan panjang
bervariasi — bukan batch seragam yang akan menyembunyikan bug penanganan padding.

In [ ]:
for enc in ["mamba2", "gru"]:
    torch.manual_seed(0)
    m = PDClassifier(encoder=enc, patch_size=20).to(DEV).train()
    x = torch.randn(4, 1880, 6, device=DEV)
    lens = torch.tensor([1880, 1420, 1180, 640], device=DEV)

    logits, alpha = m(x, lens)
    # class-weighted BCE sesuai Subbab 3.6.1: pos_weight = 62/15
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        logits, torch.tensor([1.0, 0.0, 1.0, 1.0], device=DEV),
        pos_weight=torch.tensor(62 / 15, device=DEV),
    )
    loss.backward()

    grads = [p.grad for p in m.parameters() if p.grad is not None]
    n_total = sum(1 for _ in m.parameters())
    print(f"{enc:7s} loss={loss.item():.4f}  grad {len(grads)}/{n_total} tensor  "
          f"NaN={any(torch.isnan(g).any() for g in grads)}  "
          f"Inf={any(torch.isinf(g).any() for g in grads)}  "
          f"norma_maks={max(g.abs().max().item() for g in grads):.3e}")

mamba2  loss=2.3114  grad 69/69 tensor  NaN=False  Inf=False  norma_maks=1.416e+00
gru     loss=2.4039  grad 39/39 tensor  NaN=False  Inf=False  norma_maks=1.468e+00


## 5. Ringkasan

| Butir | Hasil |
|---|---|
| Total parameter | mamba2 282.577, gru 273.025 — keduanya di bawah 500rb |
| Penyetaraan encoder | selisih −4,1%, dalam toleransi ±5% |
| Kebocoran padding (Uji A, isi padding) | tidak berpengaruh sama sekali pada kedua encoder |
| Stabilitas komposisi batch (Uji B) | mamba2 ~1e-8, gru ~1e-5 (derau float32 cuDNN, bukan bocor) |
| Bobot alpha di padding | tepat nol |
| Uji regresi arah mundur | selisih terbukti nyata, penanganan diperlukan |
| Kasus tepi 29–467 patch | seluruhnya jalan, alpha berjumlah 1, tanpa NaN |
| Forward + backward | gradien penuh, tanpa NaN/Inf |

**Catatan untuk pembahasan hasil nanti.** Panjang urutan yang dilihat encoder berkisar 29–467
token. Subbab 2.2.4 memprediksi keunggulan Mamba-2 hanya relevan pada urutan panjang, dan
memperkirakan keunggulan itu tidak muncul pada resolusi patch kasar. Rentang yang terukur di sini
menunjukkan bahwa **seluruh** resolusi yang akan diuji tergolong pendek untuk ukuran state space
model, sehingga prediksi tersebut berpeluang besar berakhir null di semua konfigurasi — bukan
hanya pada resolusi kasar. Ini bukan kegagalan rancangan, melainkan konteks yang perlu dinyatakan
di awal agar hasil null dibaca sebagai temuan yang diantisipasi, bukan sebagai anomali.

Langkah berikutnya bukan pelatihan, melainkan Tahap 2 (prapemrosesan data nyata), karena model ini
baru diuji dengan tensor acak.